# data_processor.py — unit tests

Tests three stages of the data pipeline in isolation:

| Cell | What it tests |
|------|--------------|
| Setup (1-4) | paths, processor, dataset init |
| `__getitem__` | single sample shapes + sequence structure |
| `_build_messages` | raw PIL images + chat turn structure |
| `FlattenedDataCollator` | batch assembly + cu_seqlens conversion |

In [9]:
import sys
from pathlib import Path

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "qwen-vl-finetune"))
print(f"Project root: {project_root}")

Project root: /home/rithvik/IROS_proj/cvpr_proj/Qwen3-VL


In [10]:
import os
from dataclasses import dataclass

# ---------------------------------------------------------------------------
# Dataset switcher — change DATASET_USE to any key or comma-separated keys
# ---------------------------------------------------------------------------
NAVILA_BASE = "/home/rithvik/IROS_proj/NaVILA-Dataset"

DATASET_PATHS = {
    "r2r":    f"{NAVILA_BASE}/R2R/annotations.json",
    "human":  f"{NAVILA_BASE}/Human/annotations.json",
    "rxr":    f"{NAVILA_BASE}/RxR/annotations.json",
    "scanqa": f"{NAVILA_BASE}/ScanQA/annotations/ScanQA_v1.0_train_reformat.json",
}

DATASET_USE = "r2r"      # <-- change this: "r2r" | "human" | "rxr" | "scanqa" | "r2r,rxr"
MODEL_TYPE  = "qwen3vl"  # <-- change this: "qwen2vl" | "qwen2.5vl" | "qwen3vl"

print("Available datasets:")
for name, path in DATASET_PATHS.items():
    exists = os.path.exists(path)
    marker = "OK     " if exists else "MISSING"
    active = "  <-- selected" if name in DATASET_USE.split(",") else ""
    print(f"  [{marker}] {name:8s}  {path}{active}")

# ---------------------------------------------------------------------------

@dataclass
class MockDataArgs:
    dataset_use: str = DATASET_USE
    data_flatten: bool = False
    data_packing: bool = False
    model_type: str = MODEL_TYPE
    max_pixels: int = 28 * 28 * 576
    min_pixels: int = 28 * 28 * 16
    video_max_frames: int = 8
    video_min_frames: int = 4
    video_max_pixels: int = 1024 * 28 * 28
    video_min_pixels: int = 256 * 28 * 28
    video_max_total_pixels: int = 1664 * 28 * 28
    video_min_total_pixels: int = 256 * 28 * 28
    video_fps: float = 2.0
    motion_token_text: str = "<motion>"
    gru_history_slots: int = 8
    gru_max_insert_tokens: int = 64
    gru_min_seq_len: int = 1
    gru_fallback_to_stop: bool = True
    base_interval: int = 2
    debug_motion_tokenization: bool = False

data_args = MockDataArgs()
print(f"\ndataset_use: {data_args.dataset_use}")
print(f"model_type:  {data_args.model_type}")

Available datasets:
  [OK     ] r2r       /home/rithvik/IROS_proj/NaVILA-Dataset/R2R/annotations.json  <-- selected
  [OK     ] human     /home/rithvik/IROS_proj/NaVILA-Dataset/Human/annotations.json
  [OK     ] rxr       /home/rithvik/IROS_proj/NaVILA-Dataset/RxR/annotations.json
  [OK     ] scanqa    /home/rithvik/IROS_proj/NaVILA-Dataset/ScanQA/annotations/ScanQA_v1.0_train_reformat.json

dataset_use: r2r
model_type:  qwen3vl


In [11]:
from transformers import AutoProcessor

MODEL_PATH = "/home/rithvik/NaVILA-Bench/Qwen-Model/sft_base_qwen3vl"
# MODEL_PATH = "/home/rithvik/IROS_proj/cvpr_proj/Qwen3-VL/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_PATH)
print(f"Tokenizer vocab size: {processor.tokenizer.vocab_size}")
print(f"Image processor type: {type(processor.image_processor).__name__}")

Tokenizer vocab size: 151643
Image processor type: Qwen2VLImageProcessor


In [12]:
from qwenvl.data.data_processor import LazySupervisedDataset

dataset = LazySupervisedDataset(processor, data_args)
print(f"Total samples: {len(dataset)}")

Total samples: 353894


In [13]:
import torch
import numpy as np

# ── change INDEX to inspect a different sample ──────────────────────────────
INDEX = 0
# ────────────────────────────────────────────────────────────────────────────

sample    = dataset[INDEX]
tokenizer = processor.tokenizer
input_ids = sample["input_ids"][0].tolist()
labels    = sample["labels"]

print("=== __getitem__ output ===\n")
for k, v in sample.items():
    if hasattr(v, "shape"):
        print(f"  {k:20s} shape={list(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:20s} {v}")

seq_len = sample["attention_mask"][0]
print(f"\nattention_mask = [{seq_len}]  (not a bool mask — collator converts to cu_seqlens [0, {seq_len}])")

total  = labels.numel()
active = (labels != -100).sum().item()
print(f"\nlabels: {active}/{total} tokens active ({active/total:.1%})  — rest ignored (-100)")

# special token ids used to parse sequence structure
IMAGE_PAD = tokenizer.convert_tokens_to_ids("<|image_pad|>")
VIS_START = tokenizer.convert_tokens_to_ids("<|vision_start|>")
VIS_END   = tokenizer.convert_tokens_to_ids("<|vision_end|>")
IM_START  = tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END    = tokenizer.convert_tokens_to_ids("<|im_end|>")

segments = []
i = 0
while i < len(input_ids):
    tid = input_ids[i]
    if tid == VIS_START:
        j = i + 1
        while j < len(input_ids) and input_ids[j] == IMAGE_PAD:
            j += 1
        segments.append(f"[IMAGE: {j - i - 1} patches]")
        i = j + 1  # skip VIS_END
    elif tid in (IM_START, IM_END):
        segments.append(tokenizer.decode([tid]))
        i += 1
    else:
        j = i
        while j < len(input_ids) and input_ids[j] not in (VIS_START, IMAGE_PAD, IM_START, IM_END):
            j += 1
        text = tokenizer.decode(input_ids[i:j], skip_special_tokens=True).strip()
        if text:
            segments.append(f'"{text}"')
        i = j

print("\n--- Sequence structure (text collapsed, images shown as patch counts) ---")
for seg in segments:
    print(" ", seg)

label_ids = [tid if tid != -100 else tokenizer.pad_token_id for tid in labels[0].tolist()]
print("\n--- LABEL ---")
print(tokenizer.decode(label_ids, skip_special_tokens=True))

=== __getitem__ output ===

  input_ids            shape=[1, 559]  dtype=torch.int64
  attention_mask       [559]
  mm_token_type_ids    shape=[1, 559]  dtype=torch.int64
  pixel_values         shape=[1568, 1536]  dtype=torch.float32
  image_grid_thw       shape=[8, 3]  dtype=torch.int64
  labels               shape=[1, 559]  dtype=torch.int64
  position_ids         shape=[3, 1, 559]  dtype=torch.int64

attention_mask = [559]  (not a bool mask — collator converts to cu_seqlens [0, 559])

labels: 13/559 tokens active (2.3%)  — rest ignored (-100)

--- Sequence structure (text collapsed, images shown as patch counts) ---
  <|im_start|>
  "system
You are a helpful navigation assistant."
  <|im_end|>
  <|im_start|>
  "user
Imagine you are a robot programmed for navigation tasks. You have been given a video of historical observations:"
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  "

In [14]:
from qwenvl.data.data_processor import _build_messages
from pathlib import Path

# reuses INDEX defined in the cell above
ann       = dataset.list_data_dict[INDEX]
base_path = Path(ann["data_path"])
messages, has_missing = _build_messages(ann, base_path)

print(f"has_missing: {has_missing}  |  turns: {len(messages)}\n")
for turn in messages:
    print(f"[{turn['role']}]")
    for item in turn["content"]:
        if item["type"] == "text":
            print(f"  text : {item['text'][:120]}")
        else:
            img = item["image"]
            print(f"  image: {img.size}  {img.mode}  {type(img).__name__}")
    print()

has_missing: False  |  turns: 3

[system]
  text : You are a helpful navigation assistant.

[user]
  text : Imagine you are a robot programmed for navigation tasks. You have been given a video of historical observations:
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  text : and current observation:
  image: (512, 512)  RGB  Image
  text : Your assigned task is: Turn and walk away from the bathroom into the bedroom area. Walk through the open door on the oth

[assistant]
  text : The next action is move forward 75 cm.



In [15]:
from qwenvl.data.data_processor import FlattenedDataCollatorForSupervisedDataset

# take two consecutive samples to simulate a batch of size 2
instances = [dataset[INDEX], dataset[INDEX + 1]]

print("=== Before collation (per-sample) ===")
for idx, inst in enumerate(instances):
    seq = inst["attention_mask"][0]
    print(f"\n  sample {idx}:")
    print(f"    input_ids    {list(inst['input_ids'].shape)}")
    print(f"    labels       {list(inst['labels'].shape)}")
    print(f"    position_ids {list(inst['position_ids'].shape)}")
    print(f"    pixel_values {list(inst['pixel_values'].shape)}")
    print(f"    attention_mask (seq len) = {seq}")

collator = FlattenedDataCollatorForSupervisedDataset(tokenizer=processor.tokenizer)
batch = collator(instances)

print("\n=== After collation (batch) ===\n")
for k, v in batch.items():
    if v is None:
        print(f"  {k:20s} None")
    elif hasattr(v, "shape"):
        print(f"  {k:20s} shape={list(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:20s} {v}")

# show the cu_seqlens conversion explicitly
lens = [inst["attention_mask"][0] for inst in instances]
print(f"\nper-sample lengths : {lens}")
print(f"cu_seqlens         : {batch['attention_mask'].tolist()}")
print(f"  → FlashAttention sees 2 independent sequences inside one packed tensor")

=== Before collation (per-sample) ===

  sample 0:
    input_ids    [1, 559]
    labels       [1, 559]
    position_ids [3, 1, 559]
    pixel_values [1568, 1536]
    attention_mask (seq len) = 559

  sample 1:
    input_ids    [1, 523]
    labels       [1, 523]
    position_ids [3, 1, 523]
    pixel_values [1568, 1536]
    attention_mask (seq len) = 523

=== After collation (batch) ===

  input_ids            shape=[1, 1082]  dtype=torch.int64
  labels               shape=[1, 1082]  dtype=torch.int64
  attention_mask       shape=[3]  dtype=torch.int32
  position_ids         shape=[3, 1, 1082]  dtype=torch.int64
  pixel_values         shape=[3136, 1536]  dtype=torch.float32
  image_grid_thw       shape=[16, 3]  dtype=torch.int64
  pixel_values_videos  None
  video_grid_thw       None

per-sample lengths : [559, 523]
cu_seqlens         : [0, 559, 1082]
  → FlashAttention sees 2 independent sequences inside one packed tensor
